# Relatório Final — Agente Imobiliário Multiagente (FIAP)

Este notebook documenta a **arquitetura**, o **funcionamento** e evidências do sistema desenvolvido para o Projeto Final.

**Stack:** LangGraph + LangChain · Ollama (`llama3.1:8b` / HF Meta-Llama-3.1-8B-Instruct) · ChromaDB · Streamlit · SQLite

## 1. Objetivo

Demonstrar um ecossistema multiagente capaz de:
- Atender leads com especialistas (busca, avaliação, jurídico, financiamento, vendas, CRM)
- Aprimorar respostas com **RAG** seedado
- Catalogar origem e preferências do lead
- Reengajar leads inativos
- Agendar visitas com e-mail/ICS
- Emitir relatórios gerenciais do funil

## 2. Arquitetura

```
Us (Streamlit)
    → Supervisor (roteia intenção)
        → Busca / Avaliação / Jurídico / Financiamento / Vendas / CRM / Agendamento
            → RAG (Chroma) + Tools (SQLite catálogo/CRM/agenda)
        → Síntese → resposta final
```

Insira aqui o diagrama exportado ou prints em `../assets/screenshots/`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src.config import OLLAMA_MODEL, CHROMA_DIR, DB_PATH
from src.db.models import init_db
from src.db import repository as repo

init_db()
print("Modelo:", OLLAMA_MODEL)
print("DB:", DB_PATH)
print("Chroma:", CHROMA_DIR)

## 3. Catálogo seedado (100 imóveis)

In [ ]:
resid = repo.list_properties(segmento="residencial", limit=100)
empres = repo.list_properties(segmento="empresarial", limit=100)
print(f"Residenciais: {len(resid)}")
print(f"Empresariais: {len(empres)}")
print("Exemplo residencial:", repo.property_to_dict(resid[0])["titulo"] if resid else None)
print("Exemplo empresarial:", repo.property_to_dict(empres[0])["titulo"] if empres else None)

## 4. Funil de leads e origens

In [ ]:
print("Funil:", repo.funnel_counts())
print("Origens:", repo.origem_distribution())
print("Top bairros:", repo.top_bairros_buscados())
inativos = repo.inactive_leads_for_reengagement()
print(f"Leads elegíveis a reengajamento: {len(inativos)} →", [l.id for l in inativos])

## 5. RAG — recuperação de contexto

Requer Ollama com `nomic-embed-text` e índice criado via `scripts/seed_and_ingest.py`.

In [ ]:
try:
    from src.rag.retriever import retrieve, format_context
    docs = retrieve("documentação para compra de imóvel ITBI", k=3)
    print(format_context(docs))
except Exception as e:
    print("RAG indisponível (rode o seed com Ollama):", e)

## 6. Execução ponta a ponta do grafo (exemplo)

Requer Ollama com `llama3.1:8b`. Sem o modelo, o sistema usa fallbacks heurísticos onde possível.

In [ ]:
from src.graph.workflow import run_agent

lead_id = "LEAD-001"
try:
    result = run_agent(
        "Quero um apartamento em Moema até R$ 900000",
        lead_id=lead_id,
    )
    print("Intent:", result.get("intent"))
    print("Rota:", result.get("route"))
    print("\nResposta final:\n", result.get("final_response"))
    print("\nImóveis:", [p["id"] for p in (result.get("properties_found") or [])])
except Exception as e:
    print("Falha ao executar grafo:", e)

## 7. Simulação de financiamento e agendamento (tools)

In [ ]:
from src.tools.mortgage_simulator import simulate_mortgage, format_simulation
from src.tools.scheduling import schedule_visit_or_consult
from datetime import datetime, timedelta

sim = simulate_mortgage(850_000, entrada_percent=20, prazo_meses=240)
print(format_simulation(sim))

inicio = (datetime.utcnow() + timedelta(days=2)).replace(hour=14, minute=0, second=0, microsecond=0)
appt = schedule_visit_or_consult(
    lead_id="LEAD-002",
    seller_id="SEL-001",
    inicio_iso=inicio.isoformat(),
    tipo="visita",
    property_id="RES-001",
    send_email=True,
)
print("\nAgendamento:", appt)

## 8. Relatório gerencial (métricas + narrativa)

In [ ]:
from src.services.reports import build_report

report = build_report(periodo="mes", with_narrative=True)
for k in ["leads_gerados", "leads_interessados", "leads_negociacao", "leads_comprados_pos_venda"]:
    print(f"{k}: {report[k]}")
print("\nNarrativa:\n", report.get("narrative"))

## 9. Prints da interface (entrega)

Cole/exiba imagens de `../assets/screenshots/`:
- Chat com painel de agentes/RAG
- Catálogo residencial e empresarial
- Detalhe do imóvel com galeria
- CRM de leads
- Calendário
- Relatório gerencial
- Arquivo `.ics` / `.eml` em `data/outbox/`

In [ ]:
from IPython.display import Image, display
from pathlib import Path

shots = sorted((ROOT / "assets" / "screenshots").glob("*.png"))
if not shots:
    print("Nenhum PNG em assets/screenshots — adicione os prints da demo.")
else:
    for p in shots:
        print(p.name)
        display(Image(filename=str(p)))

## 10. Limitações e próximos passos

- Dados 100% sintéticos (sem APIs reais de portais)
- Qualidade das respostas depende do modelo Ollama local
- E-mail em modo `mock` por padrão
- Evoluções: autenticação, persistência cloud, fine-tune PT-BR, integração CRM real